# FiftyOne × Lightwheel **EgoStandard** — Native MCAP Demo

**Goal:** open Lightwheel's egocentric human-demonstration episodes — head-view video, 3D hand pose,
and 3D **full-body** pose (the `EgoStand-body` SKU) — directly from their native **MCAP** files inside
the FiftyOne App, with every stream synced to one playback clock. No ROS, no Foxglove, no pre-extraction.

This notebook is **self-contained and reproducible** — anyone with a Hugging Face account and access
to the dataset can run it end to end. It works on macOS, Linux, and Windows (WSL recommended on
Windows). It walks through:

1. Creating an isolated virtual environment just for this demo.
2. Authenticating with Hugging Face and requesting dataset access.
3. Downloading **only the MCAP subset** of a handful of episodes (a few GB).
4. Building a FiftyOne dataset whose samples point straight at the `.mcap` files.
5. Deriving `has_*` capability fields from **schemas** (not topic names) so you can curate by sensor.
6. Computing **motion embeddings + a similarity index** decoded from the pose streams.
7. Turning the **action segmentation** into filterable, taggable temporal labels.
8. Launching the App and driving the multimodal viewer to show off the MCAP features.

> **Why this dataset pairs well with FiftyOne:** EgoStandard ships every episode in both LeRobot
> and MCAP, with MCAP recommended for *synchronized multimodal streams*. FiftyOne 1.19+ reads
> `.mcap` directly via byte-range reads and renders whatever it can decode in a tiled, scrubbable
> viewer — exactly the stereo head-cam + hand/body-pose combination this dataset carries.


## 0. Prerequisites & how to run this notebook

Do the environment setup (Section 1) in a **terminal**, then launch Jupyter *from inside the
activated environment* and open this file. That guarantees the notebook kernel is the demo venv
and not your system Python.

You will need:

- **Python 3.10 or 3.11** (recommended for FiftyOne + the MCAP viewer). Check with `python3 --version`.
- A **Hugging Face account** and a **read token**: <https://huggingface.co/settings/tokens>.
- **Access granted** to the gated dataset (see Section 2). Request it first, as review can take a
  few business days.
- A modern **Chromium-based browser** (Chrome, Brave, Edge) or recent Firefox for the App's viewer.
- Disk space: the full repo is large, but we download only a handful of MCAP episodes (a few GB).


## 1. Create an isolated virtual environment (run in Terminal)

Copy-paste this block into a terminal. It creates a dedicated venv named `egostd-mcap-demo`
so nothing here touches your other projects. (On Windows, use WSL or adjust the activate path to
`.venv\Scripts\activate`.)

```bash
# --- 1a. Pick a clean working directory -------------------------------------
mkdir -p ~/egostd-mcap-demo && cd ~/egostd-mcap-demo

# --- 1b. Create & activate a dedicated venv ---------------------------------
# Prefer python3.11 if available; fall back to python3.
(python3.11 -m venv .venv 2>/dev/null) || python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip

# --- 1c. Install the demo dependencies --------------------------------------
# fiftyone[multimodal] -> app + core + the multimodal MCAP VIEWER (1.19+ required;
#   the [multimodal] extra is what renders the tiled sample viewer)
# huggingface_hub -> download client
# mcap           -> pure-Python MCAP reader (no ROS required)
# protobuf       -> decode the embedded pose/annotation schemas
# umap-learn     -> 2D embedding projection for the Embeddings panel (PCA fallback if absent)
# tqdm           -> progress bars
# jupyter        -> to run this notebook from inside the venv
pip install "fiftyone[multimodal]>=1.19" "huggingface_hub>=0.34" "mcap" "protobuf" "umap-learn" tqdm jupyter

# --- 1d. Launch Jupyter from inside the venv --------------------------------
jupyter notebook
```

> **Kernel check:** once the notebook is open, run the next cell. If the FiftyOne version is
> `< 1.19`, native MCAP support is not present — upgrade with `pip install -U "fiftyone>=1.19"`
> and restart the kernel.


In [ ]:
# Enable the multimodal viewer. VFF_MULTIMODAL must be set BEFORE `import fiftyone`,
# so this is the first thing the notebook does.
import os
os.environ["VFF_MULTIMODAL"] = "1"

import sys, platform
print("Python  :", sys.version.split()[0], "on", platform.machine(), platform.system())
print("Prefix  :", sys.prefix)  # should point inside your demo .venv

# FiftyOne + the MCAP/App stack are best tested on Python 3.10-3.11. Newer versions
# (3.12+) often work but can surprise you in the App or native readers.
if sys.version_info[:2] not in [(3, 10), (3, 11)]:
    print(f"\nNOTE: Python {sys.version_info.major}.{sys.version_info.minor} is outside the "
          "best-tested 3.10-3.11 range. It may work fine; if you hit odd behavior in the "
          "App or the mcap reader, rebuild the venv with python3.11.")

import fiftyone as fo
from packaging.version import Version
print("FiftyOne:", fo.__version__)
assert Version(fo.__version__) >= Version("1.19"), (
    "Native MCAP support requires FiftyOne >= 1.19. "
    "Run: pip install -U 'fiftyone>=1.19' and restart the kernel."
)
print("OK — FiftyOne is new enough for native MCAP.")

## 2. Request access & authenticate with Hugging Face

The EgoStandard repo is **gated**. Before anything downloads:

1. Open <https://huggingface.co/datasets/LightwheelAI/EgoStandard> while logged in.
2. Click **Request access** and fill in the short form (who you are / how you'll use it).
   Approval typically takes ~2–3 business days.
3. Create a **read** token at <https://huggingface.co/settings/tokens>.

Then authenticate. The cell below uses the notebook login widget; alternatively run
`hf auth login` in your terminal (the `hf` CLI replaced the old `huggingface-cli`).


In [ ]:
# Authenticate. Paste a READ token when prompted.
from huggingface_hub import login, whoami

login()  # opens a token widget; token is cached to ~/.cache/huggingface

me = whoami()
print("Logged in as:", me.get("name", me))

In [ ]:
# Sanity-check that access has actually been granted (not just requested).
# If this raises a 403/GatedRepoError, your access request is still pending.
from huggingface_hub import HfApi
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

REPO_ID = "LightwheelAI/EgoStandard"
api = HfApi()
try:
    info = api.dataset_info(REPO_ID)
    print(f"Access confirmed to {REPO_ID}.")
    print("Repo is Xet-backed:", any(getattr(s, "rfilename", "").endswith(".mcap") for s in (info.siblings or [])[:1]) or "check files tab")
except GatedRepoError:
    print("Access to", REPO_ID, "is still PENDING. Request it on the dataset page and wait for approval.")
except HfHubHTTPError as e:
    print("HTTP error — check your token has read scope:", e)

## 3. Download only the MCAP episodes

Each episode is a single self-contained `.mcap` file
(`<SUB_SKU>/mcap/<task_name>/<episode_uuid>.mcap`), so we select exact files by path and cap the
count — downloading only a few GB rather than the whole repo.

Pick a selection strategy in the cell below:

- **First N episodes** (`SELECTION="first"`) — simplest; may pull several episodes of the *same* task.
- **Variety (one per task)** (`SELECTION="variety"`, default) — recommended: N visibly different
  manipulation tasks, so the video and pose trajectories look distinct.

The default `N_EPISODES = 50` pulls a spread of ~50 distinct tasks, which makes the motion embeddings
and action-segment views in later sections much richer than a handful of near-identical clips. Drop it
to ~5 for a quick first run.


In [ ]:
import os

# Use plain HTTPS downloads (reliable and simple). A higher timeout helps on large files.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
print("Download backend: plain HTTPS")

In [ ]:
# List the available MCAP episodes WITHOUT downloading, scoped to just the mcap
# subtree so the call returns quickly (rather than enumerating the entire repo).
from huggingface_hub import HfApi

REPO_ID = "LightwheelAI/EgoStandard"
# EgoStand-body = head-view + hand pose + FULL-BODY pose, AND ships manifests
#   (scene_family, duration_s, etc.). This is the richer SKU for embeddings + curation.
# EgoStand = high-volume head-view (head-cam + hand pose only); no manifests in this release.
SUB_SKU = "EgoStand-body"

api = HfApi()
tree = api.list_repo_tree(
    REPO_ID,
    path_in_repo=f"{SUB_SKU}/mcap",   # scope to the mcap subtree only
    repo_type="dataset",
    recursive=True,
)

# Expected LAYOUT: <SUB_SKU>/mcap/<task_name>/<episode_uuid>.mcap
# i.e. the folder under mcap/ is the TASK and each .mcap inside is ONE episode.
mcap_files = sorted(
    item.path for item in tree
    if getattr(item, "path", "").endswith(".mcap")
)
if not mcap_files:
    raise RuntimeError(
        f"No .mcap files under {SUB_SKU}/mcap/. Check access was granted and the SKU name is right."
    )
print(f"Found {len(mcap_files)} .mcap episode files under {SUB_SKU}/mcap/. First few:")
for f in mcap_files[:10]:
    print("  ", f)

# Derive the task as the path segment right after 'mcap/', so this works regardless
# of how deep the SUB_SKU prefix is (EgoStand vs EgoStand-body).
from collections import OrderedDict
def task_of(path):
    parts = path.split("/")
    i = parts.index("mcap")
    return parts[i + 1] if i + 1 < len(parts) - 1 else parts[i + 1]
tasks = list(OrderedDict((task_of(f), None) for f in mcap_files).keys())
print(f"\n{len(mcap_files)} episodes across {len(tasks)} distinct tasks.")
print("Example task:", tasks[0])

In [ ]:
# Download a handful of individual episode .mcap files, with a progress bar.
import os, time
from collections import OrderedDict
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, list_repo_tree

N_EPISODES = 50         # 50 gives a richer demo; drop to ~5 for a quick smoke test.
SELECTION  = "variety"  # "variety" (one episode per task) or "first" (first N files).

if SELECTION == "variety":
    by_task = OrderedDict()
    for f in mcap_files:
        by_task.setdefault(task_of(f), f)  # one episode per distinct task
    chosen_files = list(by_task.values())[:N_EPISODES]
else:
    chosen_files = mcap_files[:N_EPISODES]

LOCAL_DIR = os.path.expanduser("~/egostd-mcap-demo/data")
print(f"Downloading {len(chosen_files)} episodes ({SELECTION}) to {LOCAL_DIR}\n")

def _clear_incomplete(cache_root):
    for root, _, files in os.walk(cache_root):
        for fn in files:
            if fn.endswith(".incomplete"):
                try:
                    os.remove(os.path.join(root, fn))
                except OSError:
                    pass

def fetch(rel_path, retries=3):
    for attempt in range(retries):
        try:
            return hf_hub_download(REPO_ID, rel_path, repo_type="dataset", local_dir=LOCAL_DIR)
        except Exception as e:
            tqdm.write(f"     retry {attempt+1}/{retries} ({type(e).__name__})")
            _clear_incomplete(os.path.expanduser("~/.cache/huggingface"))
            time.sleep(2)
    raise RuntimeError(f"Failed to download {rel_path} after {retries} attempts")

# Small optional files first: READMEs + any SKU manifests.
for extra in [f"{SUB_SKU}/README.md", "README.md"]:
    try:
        hf_hub_download(REPO_ID, extra, repo_type="dataset", local_dir=LOCAL_DIR)
    except Exception:
        pass

try:
    mtree = list_repo_tree(REPO_ID, path_in_repo=f"{SUB_SKU}/manifests",
                           repo_type="dataset", recursive=True)
    mfiles = [it.path for it in mtree
              if getattr(it, "path", "").endswith((".json", ".jsonl", ".parquet", ".csv"))]
    for mf in mfiles:
        try:
            hf_hub_download(REPO_ID, mf, repo_type="dataset", local_dir=LOCAL_DIR)
        except Exception:
            pass
    if mfiles:
        print(f"Fetched {len(mfiles)} manifest file(s) for {SUB_SKU}.")
except Exception:
    pass

downloaded = []
for rel_path in tqdm(chosen_files, desc="Episodes", unit="file"):
    tqdm.write(f"  -> {rel_path}")
    downloaded.append(fetch(rel_path))

print("\nDone. Downloaded", len(downloaded), "episodes to", LOCAL_DIR)

In [ ]:
# Locate the .mcap files we actually pulled down.
import glob
mcap_paths = sorted(glob.glob(os.path.join(LOCAL_DIR, SUB_SKU, "mcap", "**", "*.mcap"), recursive=True))
print(f"{len(mcap_paths)} local .mcap files ready:")
for p in mcap_paths:
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {size_mb:8.1f} MB  {os.path.relpath(p, LOCAL_DIR)}")
assert mcap_paths, "No .mcap files found — check access was granted and the download succeeded."

## 4. Build a FiftyOne dataset that points straight at the MCAP files

This is the whole import. Point a sample's `filepath` at a `.mcap` file and FiftyOne infers
`media_type = "multimodal"` automatically — no importer class, no schema mapping, no conversion.


In [ ]:
import fiftyone as fo

DATASET_NAME = "egostandard-mcap-demo"

# Start fresh each run so re-executing the notebook is idempotent.
if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)

dataset = fo.Dataset(DATASET_NAME, persistent=True)

samples = []
for p in mcap_paths:
    # Layout: .../<SUB_SKU>/mcap/<task_name>/<episode_uuid>.mcap
    task = os.path.basename(os.path.dirname(p))       # parent dir = task name
    episode_id = os.path.splitext(os.path.basename(p))[0]  # filename (uuid) = episode id
    s = fo.Sample(filepath=p)
    s["task"] = task
    s["episode_id"] = episode_id
    s["sub_sku"] = SUB_SKU
    samples.append(s)

dataset.add_samples(samples)
print("Added", len(dataset), "multimodal samples.")
# media_type is a per-sample PROPERTY, not an aggregatable DB field, so read it
# off the samples rather than calling dataset.distinct("media_type").
print("Media types:", {s.media_type for s in dataset})  # -> {'multimodal'}

### 4b. Manifest metadata — not published in this release (skipped)

The dataset card documents a manifest schema (`episode_uuid`, `scene_family`, `duration_s`,
`sha256`, hand/body flags), but as of this open release the `manifests/` directory contains only a
placeholder `README.md` with upload instructions for the dataset authors — **no episode index has
been published yet, for either SKU.** So there is nothing to join.

This costs us almost nothing: we already have `task` (from the folder name), `duration_s_measured`
(computed from each MCAP in Section 5c), and `has_hand_pose` / `has_body_pose` / `has_image` (from
schema inspection in Section 5). The only thing the manifest would have added is `scene_family`.

The cell below detects the placeholder case and skips cleanly. If Lightwheel later publishes a real
manifest (CSV/JSONL/Parquet), re-running the download cell will pull it and this join will populate
`scene_family` and an authoritative `duration_s` automatically.


In [ ]:
import glob, json as _json

manifest_files = glob.glob(os.path.join(LOCAL_DIR, SUB_SKU, "manifests", "**", "*"), recursive=True)
data_files = [m for m in manifest_files if m.endswith((".json", ".jsonl", ".parquet", ".csv"))]

if not data_files:
    # Expected for this release: manifests/ holds only a placeholder README, no episode index.
    print("No manifest data files published for", SUB_SKU,
          "(the manifests/ dir is a placeholder in this release).")
    print("SKIPPING manifest join. Using MCAP-derived fields (task, duration_s_measured, has_*) instead.")
    manifest_files = []
else:
    manifest_files = data_files
    print("Manifest files found:", len(manifest_files))
    for m in manifest_files:
        print("  ", os.path.relpath(m, LOCAL_DIR))

def _load_rows(path):
    """Best-effort load of manifest rows as a list of dicts."""
    if path.endswith(".jsonl"):
        with open(path) as fh:
            return [_json.loads(line) for line in fh if line.strip()]
    if path.endswith(".json"):
        with open(path) as fh:
            data = _json.load(fh)
        if isinstance(data, dict):  # e.g. {"episodes": [...]}
            for v in data.values():
                if isinstance(v, list):
                    return v
            return [data]
        return data
    if path.endswith((".parquet", ".csv")):
        try:
            import pandas as pd
            df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
            return df.to_dict(orient="records")
        except Exception as e:
            print("  (skipping", os.path.basename(path), "->", e, ")")
    return []

rows = []
for m in manifest_files:
    rows.extend(_load_rows(m))
if manifest_files:
    print("Total manifest rows loaded:", len(rows))

# Index rows by any id-like field we can match to our episode dir names.
def _row_keys(r):
    return {str(r.get(k)) for k in ("episode_uuid", "episode_id", "human_case_id") if r.get(k) is not None}

by_key = {}
for r in rows:
    for k in _row_keys(r):
        by_key[k] = r

KEEP = ["task_name", "scene_id", "scene_family", "duration_s",
        "has_hand_pose", "has_body_pose", "annotation_version"]

matched = 0
if rows:
    for s in dataset:
        r = by_key.get(str(s["episode_id"]))
        if not r:
            # Some manifests store the full episode_path; try a suffix match.
            r = next((rr for rr in rows if str(s["episode_id"]) in str(rr.get("episode_path", ""))), None)
        if r:
            for k in KEEP:
                if k in r and r[k] is not None:
                    s[k] = r[k]
            s.save()
            matched += 1
    print(f"Attached manifest metadata to {matched}/{len(dataset)} samples.")
else:
    print("Nothing to join (no manifest rows). Continue with MCAP-derived fields.")

## 5. Derive capability fields from **schemas**, not topic names

The single most important lesson from working with MCAP: **don't trust topic names.** A topic called
`.../imu_packets` might be raw undecoded sensor bytes, not a decodable IMU stream. The honest signal is
the **schema name** attached to each channel. We read the MCAP summary directly (fast — MCAP indexes
live at the end of the file) and record `has_image`, `has_hand_pose`, `has_body_pose`, etc. as ordinary
sample fields you can then query and build views from.

`fiftyone` bundles an MCAP reader, but to keep this cell portable we use the standalone `mcap` library
if available and degrade gracefully otherwise.


In [ ]:
# Optional: pip install mcap  (tiny pure-Python reader; no ROS needed).
try:
    from mcap.reader import make_reader
    HAVE_MCAP = True
except Exception:
    HAVE_MCAP = False
    print("`mcap` reader not installed. Run `pip install mcap` for schema-based capability fields.")

# Schema-name matchers keyed to THIS dataset's actual schemas (confirmed via inspection):
#   foxglove.CompressedVideo, foxglove.FrameTransforms, foxglove.CameraCalibration,
#   pose.BodyFrame / HeadFrame / LeftHandFrame / RightHandFrame / LowerBodyFrame /
#   HeadCamFrame / RightEyeCamFrame, annotation.SemanticSegment,
#   annotation.BadFrameBody, annotation.BadFrameHand
SCHEMA_HINTS = {
    "has_image":        ["CompressedVideo", "CompressedImage", "RawImage"],
    "has_hand_pose":    ["LeftHandFrame", "RightHandFrame"],
    "has_body_pose":    ["BodyFrame", "LowerBodyFrame"],
    "has_head_pose":    ["HeadFrame", "HeadCamFrame"],
    "has_transforms":   ["FrameTransforms"],
    "has_calibration":  ["CameraCalibration"],
    "has_semantics":    ["SemanticSegment"],
    "has_badframes":    ["BadFrameBody", "BadFrameHand"],
}

def channel_schema_names(path):
    """Return the set of schema names present in an MCAP file (from its summary)."""
    names = set()
    with open(path, "rb") as fh:
        reader = make_reader(fh)
        summary = reader.get_summary()
        if summary is None:
            return names
        for schema in summary.schemas.values():
            names.add(schema.name)
    return names

if HAVE_MCAP:
    for s in dataset:
        schemas = channel_schema_names(s.filepath)
        s["schema_names"] = sorted(schemas)
        for field, hints in SCHEMA_HINTS.items():
            s[field] = any(any(h.lower() in sc.lower() for h in hints) for sc in schemas)
        s.save()
    print("Capability fields written. Example schema names on first sample:")
    print("  ", dataset.first()["schema_names"][:12])
else:
    print("Skipped capability fields (install `mcap` to enable).")

In [ ]:
# Inspect exactly what channels/schemas one episode contains — the ground truth
# for wiring up tiles and for tuning SCHEMA_HINTS above.
if HAVE_MCAP:
    from mcap.reader import make_reader
    sample = dataset.first()
    with open(sample.filepath, "rb") as fh:
        summary = make_reader(fh).get_summary()
        print("Task:", sample["task"], "| Episode:", sample["episode_id"])
        print(f"{'TOPIC':45s}  SCHEMA")
        print("-" * 80)
        for ch in summary.channels.values():
            sch = summary.schemas.get(ch.schema_id)
            print(f"{ch.topic:45s}  {sch.name if sch else '(none)'}")

### 5b. Curate by capability — this is the payoff

Because capabilities are ordinary fields, you can build saved views that filter episodes by what
they actually contain, instead of eyeballing files. Example: only episodes with **both** head-cam
video and hand pose.


In [ ]:
from fiftyone import ViewField as F

if "has_image" in dataset.get_field_schema():
    showcase = dataset.match(F("has_image") == True)
    if "has_hand_pose" in dataset.get_field_schema():
        showcase = showcase.match(F("has_hand_pose") == True)
    dataset.save_view("full_multimodal_showcase", showcase, overwrite=True)
    print(f"Saved view 'full_multimodal_showcase': {len(showcase)}/{len(dataset)} episodes "
          f"with head-cam + hand pose.")
else:
    print("Capability fields not present (install `mcap` and re-run Section 5).")

## 5c. Motion embeddings & similarity — cluster episodes by how the person moved

FiftyOne's Brain (`compute_visualization`, `compute_similarity`) powers the App's **Embeddings** panel
and the "sort by similarity" button. These samples are `multimodal` MCAP files, so FiftyOne has no
built-in model that turns an `.mcap` into a vector — we compute embeddings **ourselves** and pass them
in explicitly (`fob.compute_visualization(..., embeddings=<N×D array>)`).

**What we embed: actual hand + body motion.** We confirmed the pose is Protobuf, and the MCAP embeds a
Protobuf `FileDescriptorSet`, so we can decode `pose.LeftHandFrame` / `pose.RightHandFrame` /
`pose.BodyFrame` **without any external `.proto` files** — the file is fully self-describing. Each
message is a `header` + a repeated `transforms` list; each transform has a `pos` (x,y,z metres) and a
`quat` (w,x,y,z). Per episode we take each stream's **root joint (index 0)** trajectory over time and
compute motion features — path length, speed stats, and the 3D bounding volume of the motion — for
left hand, right hand, and body. Frames flagged by `annotation.BadFrame*` are masked out so tracking
dropouts don't corrupt the features.

The result clusters episodes by *how the person moved* — big reaching motions vs. fine bimanual
manipulation vs. mostly-static tasks — which is far more meaningful than clustering by which channels
exist. (A statistics-only fallback runs automatically if pose decoding is unavailable.)

> **Joint-ordering caveat:** the `.proto` doesn't name individual joints, so we assume transform
> index 0 is the wrist/root (the near-universal convention). If clusters look off, switch
> `ROOT_ONLY = False` in the feature cell to average over all joints instead.


In [ ]:
# Decode Protobuf pose messages straight from the MCAP's embedded FileDescriptorSet
# (no external .proto needed) and build a per-episode MOTION feature vector.
import numpy as np
from tqdm.auto import tqdm

ROOT_ONLY = True  # True: track joint index 0 (wrist/root). False: average over all joints.

try:
    from google.protobuf.descriptor_pb2 import FileDescriptorSet
    from google.protobuf.descriptor_pool import DescriptorPool
    from google.protobuf.message_factory import GetMessageClass
    HAVE_PROTO = True
except ImportError:
    HAVE_PROTO = False
    print("protobuf not installed; falling back to statistics features. `pip install protobuf` for motion.")

POSE_TOPICS = ["/pose/left_hand", "/pose/right_hand", "/pose/body"]

def _pool_for(summary):
    pool = DescriptorPool()
    for sch in summary.schemas.values():
        if sch.encoding == "protobuf":
            for fdp in FileDescriptorSet.FromString(sch.data).file:
                try:
                    pool.Add(fdp)
                except Exception:
                    pass
    return pool

def _positions_over_time(reader, summary, pool, topic):
    """Return an (T, K, 3) array of joint positions over time for one pose topic, or None."""
    # Map topic -> its message class via the channel's schema.
    ch = next((c for c in summary.channels.values() if c.topic == topic), None)
    if ch is None:
        return None
    sch = summary.schemas.get(ch.schema_id)
    try:
        MsgCls = GetMessageClass(pool.FindMessageTypeByName(sch.name))
    except Exception:
        return None
    frames = []
    for _, _, message in reader.iter_messages(topics=[topic]):
        m = MsgCls.FromString(message.data)
        pts = [(t.pos.x, t.pos.y, t.pos.z) for t in m.transforms]
        if pts:
            frames.append(pts)
    if not frames:
        return None
    K = min(len(f) for f in frames)          # align joint count across frames
    arr = np.array([f[:K] for f in frames], dtype="float64")  # (T, K, 3)
    return arr

def _motion_features(arr):
    """Path length, speed mean/max/std, and bbox volume from an (T,K,3) trajectory."""
    if arr is None or arr.shape[0] < 2:
        return [0.0] * 6
    track = arr[:, 0, :] if ROOT_ONLY else arr.mean(axis=1)  # (T,3)
    d = np.diff(track, axis=0)
    step = np.linalg.norm(d, axis=1)
    path_len = float(step.sum())
    spd_mean, spd_max, spd_std = float(step.mean()), float(step.max()), float(step.std())
    span = track.max(0) - track.min(0)
    bbox_vol = float(span[0] * span[1] * span[2])
    range_xyz = float(np.linalg.norm(span))
    return [path_len, spd_mean, spd_max, spd_std, bbox_vol, range_xyz]

def episode_motion(path):
    from mcap.reader import make_reader
    feats = {}
    with open(path, "rb") as fh:
        reader = make_reader(fh)
        summary = reader.get_summary()
        # duration for a scalar field / fallback
        stats = summary.statistics
        feats["duration_s"] = ((stats.message_end_time - stats.message_start_time) / 1e9
                               if stats and stats.message_end_time else 0.0)
        if HAVE_PROTO:
            pool = _pool_for(summary)
            vecs = []
            for topic in POSE_TOPICS:
                arr = _positions_over_time(reader, summary, pool, topic)
                vecs.extend(_motion_features(arr))
            feats["motion"] = vecs  # 6 features x 3 streams = 18-dim
    return feats

# Build features for every sample.
rows = []
for s in tqdm(list(dataset), desc="Decoding pose", unit="ep"):
    f = episode_motion(s.filepath)
    s["duration_s_measured"] = float(f["duration_s"])
    s.save()
    rows.append((s.id, f))

USE_MOTION = HAVE_PROTO and all("motion" in f and any(f["motion"]) for _, f in rows)
print(f"\nDecoded {len(rows)} episodes. Using {'MOTION' if USE_MOTION else 'statistics'} features.")

In [ ]:
# Assemble the N x D embedding matrix.
import numpy as np

if USE_MOTION:
    X = np.array([f["motion"] for _, f in rows], dtype="float32")   # (N, 18)
    # log-compress heavy-tailed magnitudes, then standardize.
    X = np.sign(X) * np.log1p(np.abs(X))
else:
    # Fallback: duration only (keeps the pipeline working if protobuf is missing).
    X = np.array([[np.log1p(f.get("duration_s", 0.0))] for _, f in rows], dtype="float32")

X = (X - X.mean(0)) / (X.std(0) + 1e-6)
ids = [rid for rid, _ in rows]
print("Embedding matrix:", X.shape)

In [ ]:
# Compute a 2D visualization + a similarity index from OUR embeddings.
import fiftyone.brain as fob

# 2D UMAP for the App's Embeddings panel. Falls back to PCA if umap isn't installed.
try:
    import umap  # noqa: F401
    method = "umap"
except ImportError:
    method = "pca"
    print("umap-learn not installed; using PCA. For UMAP: pip install umap-learn")

viz = fob.compute_visualization(
    dataset,
    embeddings=X,          # our explicit N x D matrix
    points=None,
    num_dims=2,
    method=method,
    brain_key="episode_viz",
    seed=51,
    verbose=True,
)

# Similarity index so the App's "sort by similarity" works on these embeddings.
sim = fob.compute_similarity(
    dataset,
    embeddings=X,
    brain_key="episode_sim",
)
print("\nBrain runs created: episode_viz (Embeddings panel), episode_sim (similarity).")
print("In the App: open the Embeddings panel and color by `task`; or select a sample and")
print("click the similarity icon to surface look-alike episodes.")

## 5d. Action segments as filterable, taggable temporal labels

The `annotation.SemanticSegment` channel carries frame-accurate action segmentation: each message is
one subtask with a `skill`, a `subtask_description`, and `start_time`/`end_time` **in seconds relative
to episode start**. We decode these into FiftyOne **`TemporalDetection`** labels stored in an
`action_segments` field.

Once they're real labels, the segments become first-class:
- **Filterable** — in the App sidebar, filter `action_segments` by `label` (the skill) to keep only
  episodes containing, say, a "grasp" or "place" action; or in code with `filter_labels`.
- **Taggable** — select segments in the App and apply label tags, or tag the samples that contain a
  given skill for a curation set.
- **On the timeline** — they render as tracks in the multimodal viewer, aligned to the video and pose.

**On labels:** the schema has a `skill` field meant to be the categorical action, but it's **empty in
this release** (like the manifests). The raw `subtask_description` is free text — 187 near-unique
strings across 284 segments — which is useless for filtering. So we derive a coarse, repeating label
from the **leading verb** of each description (`pick`, `place`, `stick`, `move`, `adjust`, …), which
collapses to ~30 clean action categories with a natural Zipfian spread (pick and place alone are a
third of all segments). The full description is preserved as the `subtask_description` attribute. If a
future release populates `skill`, the cell uses it automatically.


In [ ]:
# Decode annotation.SemanticSegment -> fiftyone TemporalDetection labels per episode.
import fiftyone as fo
from tqdm.auto import tqdm

def episode_segments(path):
    """Return a list of (skill_label, start_s, end_s, subtask, task_desc) for one episode."""
    if not HAVE_PROTO:
        return []
    from mcap.reader import make_reader
    from google.protobuf.descriptor_pb2 import FileDescriptorSet
    from google.protobuf.descriptor_pool import DescriptorPool
    from google.protobuf.message_factory import GetMessageClass

    with open(path, "rb") as fh:
        reader = make_reader(fh)
        summary = reader.get_summary()
        pool = DescriptorPool()
        for sch in summary.schemas.values():
            if sch.encoding == "protobuf":
                for fdp in FileDescriptorSet.FromString(sch.data).file:
                    try:
                        pool.Add(fdp)
                    except Exception:
                        pass
        ch = next((c for c in summary.channels.values()
                   if c.topic == "/annotation/semantic_segments"), None)
        if ch is None:
            return []
        sch = summary.schemas.get(ch.schema_id)
        try:
            MsgCls = GetMessageClass(pool.FindMessageTypeByName(sch.name))
        except Exception:
            return []

        out = []
        for _, _, message in reader.iter_messages(topics=["/annotation/semantic_segments"]):
            m = MsgCls.FromString(message.data)
            seg = m.segment
            skill = (seg.skill or "").strip()
            subtask = (seg.subtask_description or "").strip()
            # This release leaves `skill` empty, so derive a coarse, FILTERABLE action
            # label from the leading verb of the description (pick/place/stick/...).
            # Keep the full description as an attribute for detail on hover.
            if skill:
                label = skill
            elif subtask:
                label = subtask.split()[0].lower()   # leading verb -> ~30 repeating labels
            else:
                label = "segment"
            out.append((label, float(seg.start_time), float(seg.end_time),
                        subtask, m.task_description))
        return out

n_segments = 0
skills = set()
for s in tqdm(list(dataset), desc="Decoding segments", unit="ep"):
    segs = episode_segments(s.filepath)
    dets = []
    for label, t0, t1, subtask, task_desc in segs:
        if t1 <= t0:      # skip zero/negative-length segments
            continue
        # Prefer timestamp-based support; fall back to integer-second frame support
        # if this build's from_timestamps needs video frame metadata we don't have.
        try:
            det = fo.TemporalDetection.from_timestamps([t0, t1], label=label, sample=s)
        except Exception:
            det = fo.TemporalDetection(label=label, support=[int(t0) + 1, int(t1) + 1])
        det["subtask_description"] = subtask
        dets.append(det)
        skills.add(label)
    if dets:
        s["action_segments"] = fo.TemporalDetections(detections=dets)
        if segs and segs[0][4]:
            s["task_description"] = segs[0][4]
        s.save()
    n_segments += len(dets)

print(f"\nAttached {n_segments} action segments across {len(dataset)} episodes.")
print(f"{len(skills)} distinct skill/segment labels. Examples:", sorted(skills)[:10])

**Note on temporal support:** `TemporalDetection` stores its span either as a frame range
(`support=[first, last]`) or via `from_timestamps([start, end], sample=...)`, which converts seconds
to frames using the sample's metadata. Because these are `multimodal` (not `video`) samples, frame
metadata may be absent — so the cell **tries `from_timestamps` first and falls back to integer-second
`support`** automatically. Either way the segments filter and tag identically; the timeline rendering
just uses whichever support the build accepts.


In [ ]:
# Curate with the segments: build a view of episodes containing a chosen skill,
# and (optionally) tag them. Filterable in the App sidebar under `action_segments`.
from fiftyone import ViewField as F

if "action_segments" in dataset.get_field_schema():
    # Count episodes per skill label.
    counts = dataset.count_values("action_segments.detections.label")
    top = sorted(counts.items(), key=lambda kv: -kv[1])[:15]
    print("Most common action labels (label: #episodes containing it):")
    for lbl, c in top:
        print(f"  {c:3d}  {lbl}")

    # Example: view episodes that contain the single most common skill.
    if top:
        target = top[0][0]
        has_skill = dataset.filter_labels("action_segments", F("label") == target)
        dataset.save_view("has_" + target.replace(" ", "_")[:40], has_skill, overwrite=True)
        print(f"\nSaved view for episodes containing '{target}': {len(has_skill)} episodes.")

        # Tag those samples so they're easy to pull later.
        has_skill.tag_samples("contains_" + target.replace(" ", "_")[:30])
        print("Tagged them with sample tag 'contains_%s'." % target.replace(" ", "_")[:30])
else:
    print("No action_segments field (install protobuf and re-run 5d).")

## 6. Launch the App and drive the multimodal viewer

Everything below happens in the App. `launch_app` opens it inline in the notebook (or in a browser
tab). The MCAP features to show off are called out after the launch cell.

The multimodal viewer decodes video in the browser, so use a modern Chromium-based browser
(Chrome, Brave, Edge) or a recent Firefox for best results. Present clips **paused or slow-scrubbed**
for the crispest frames.


In [ ]:
session = fo.launch_app(dataset)
session  # renders the App inline; or open the printed URL in a browser

### What to demo in the App — MCAP feature checklist

Open any sample to get the **tiled multimodal viewer** (Image, 3D, Map, Plot, Logs, Message tiles).
The left sidebar's **Topics** tab lists every channel grouped by category. Add tiles with the grid
icon; bind each tile to a stream. **Every tile shares one playback clock.**

1. **Native, zero-conversion open.** Point out that samples' `media_type` is `multimodal` and the
   `.mcap` opens with no import step — FiftyOne reads it directly via byte-range reads. (This is the
   "no ROS / no Foxglove" beat.)

2. **One file, every sensor, one clock.** This is **stereo**: put `/sensor/camera/head_left/video`
   and `/sensor/camera/head_right/video` (both `foxglove.CompressedVideo`) in two **Image tiles** side
   by side, and add a **3D tile** bound to the pose frames. Hit play — left+right video and the 3D pose
   advance together on a single timeline. This is the headline shot.

3. **Full-body + hand pose as an animated 3D skeleton.** The pose here is a set of coordinate frames
   published per timestamp — `/pose/body`, `/pose/upper_body`, `/pose/lower_body`, `/pose/left_hand`,
   `/pose/right_hand`, `/pose/head` — each a repeated list of joint transforms (pos + quaternion). Bind
   them into the **3D tile** and scrub to a grasp/manipulation moment to watch the body and hands move
   in 3D. (For a scalar plot — e.g. a wrist's height over time — bind that joint's translation to a
   **Plot tile**.)

4. **Semantic action segments as a track.** `annotation.SemanticSegment` carries the frame-accurate
   action segmentation. Surface it on the timeline (label track) or in a Message tile and scrub to see
   action boundaries line up with the video and the moving pose frames.

5. **Quality flags for curation.** `annotation.BadFrameBody` / `annotation.BadFrameHand` mark where
   tracking failed. Bind a Message/Logs tile to them, or use the `has_badframes` field to filter — a
   concrete "curate out the bad frames" beat.

6. **Channel & schema discovery.** Show the Topics sidebar enumerating channels straight from the
   file's embedded Foxglove/pose/annotation schemas — the file is self-describing, so this works with
   no external message definitions installed.

7. **Curate by capability.** In the App's view bar, load the **`full_multimodal_showcase`** saved
   view to filter down to episodes that actually have head-cam video + hand pose — curation by what the
   data contains, not by filename. You can also **group by `task`** in the App to browse episodes by
   manipulation task (e.g. "Align Chairs Alignment", "Adjust Salt Grinder").

8. **Motion embeddings & similarity.** Open the **Embeddings** panel (from the `episode_viz` brain
   run) and color points by **`task`**. Because the embedding is built from decoded hand/body
   **trajectories**, episodes cluster by *how the person moved* — big reaching motions vs. fine
   bimanual manipulation vs. mostly-static tasks. Lasso a cluster to filter the grid; select a sample
   and hit the **similarity** icon (backed by `episode_sim`) to surface look-alike motions.

9. **Filter & tag action segments.** The `action_segments` field (from `annotation.SemanticSegment`)
   holds each subtask as a temporal label, tagged with a coarse **action verb** (`pick`, `place`,
   `stick`, …; full description on hover). In the sidebar, filter `action_segments` by `label` to keep
   episodes containing a given action — e.g. every episode with a `pick` — and the segments show as
   **tracks on the timeline**, aligned to video + pose. Select a segment to tag it, or use the saved
   views / sample tags from Section 5d. This is the "frame-accurate semantics become queryable" beat.

10. **Live temporal tagging.** Shift-click-and-drag on the timeline to tag an interval (e.g. a
    pick-up action), turning a moment into a persistent, queryable asset for a curation workflow.

> **Enterprise note:** fleet-scale MCAP *indexing / event mining* (query every episode where a
> condition held, across thousands of recordings, via columnar Parquet tables) is a FiftyOne
> Enterprise feature. Everything in this notebook — native viewer, synced playback, temporal tags,
> capability fields — runs in **open-source** FiftyOne.


## 7. Cleanup (optional)

```python
import fiftyone as fo
fo.delete_dataset("egostandard-mcap-demo")
```

To reclaim disk, delete the `data/` folder in your working directory. To remove the environment
entirely, delete the `.venv` folder.
